# 03 — Baseline Experiment
**SmartBiz AI V1.0 — Phase 3: AI Research Pipeline**

This notebook trains three baseline forecasting models on the preprocessed
sales data and evaluates them using MAE, RMSE, WAPE, and sMAPE.

### Models
1. Seasonal Naive (season_length=7)
2. Moving Average (window=7)
3. Exponential Smoothing (alpha=0.3)

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print('Libraries loaded.')

## 1. Prepare Data

In [ ]:
from src.data.loader import load_csv
from src.preprocessing.cleaning import clean_sales_data
from src.preprocessing.transformation import filter_product, split_train_test
from src.preprocessing.feature_engineering import engineer_features

# Load and clean
df = load_csv('../data/raw/sample_sales_data.csv')
df = clean_sales_data(df)

# Work with Product 1
df_p1 = filter_product(df, product_id=1)
df_p1 = engineer_features(df_p1)

# Train/test split
train_df, test_df = split_train_test(df_p1, test_ratio=0.2)

train_y = train_df['sales_qty'].values
test_y = test_df['sales_qty'].values

print(f'Train: {len(train_y)} | Test: {len(test_y)}')

## 2. Train Baseline Models

In [ ]:
from src.forecasting.baseline import run_baseline_experiment

predictions = run_baseline_experiment(
    train_y, test_y,
    season_length=7, ma_window=7, ses_alpha=0.3
)

for name, preds in predictions.items():
    print(f'{name}: {preds}')

## 3. Evaluate Models

In [ ]:
from src.forecasting.evaluation import evaluate_all_models

results = evaluate_all_models(test_y, predictions)

results_df = pd.DataFrame(results)
results_df

## 4. Visualize Predictions

In [ ]:
test_dates = test_df['date'].values

plt.figure(figsize=(12, 6))
plt.plot(test_dates, test_y, 'ko-', label='Actual', linewidth=2, markersize=6)

colors = ['#e74c3c', '#3498db', '#2ecc71']
for (name, preds), color in zip(predictions.items(), colors):
    plt.plot(test_dates, preds, '--', label=name, linewidth=1.5, color=color)

plt.title('Baseline Model Predictions vs Actual (Product 1)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales Qty')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/baseline_predictions.png', dpi=150)
plt.show()

## 5. Metric Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metrics = ['mae', 'rmse', 'wape', 'smape']
titles = ['MAE', 'RMSE', 'WAPE', 'sMAPE']

for ax, metric, title in zip(axes, metrics, titles):
    values = results_df[metric].values
    bars = ax.bar(results_df['model_name'], values, color=['#e74c3c', '#3498db', '#2ecc71'],
                  edgecolor='black', alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=35)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Baseline Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/plots/baseline_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Results

In [ ]:
# Save metrics as JSON
os.makedirs('../results/metrics', exist_ok=True)
with open('../results/metrics/baseline_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Metrics saved to results/metrics/baseline_metrics.json')

# Save predictions as CSV
os.makedirs('../results/predictions', exist_ok=True)
pred_df = pd.DataFrame({'date': test_df['date'].values, 'actual': test_y})
for name, preds in predictions.items():
    pred_df[name] = preds
pred_df.to_csv('../results/predictions/baseline_predictions.csv', index=False)
print('Predictions saved to results/predictions/baseline_predictions.csv')

# Save metrics table as CSV
results_df.to_csv('../results/metrics/baseline_metrics.csv', index=False)
print('Metrics table saved to results/metrics/baseline_metrics.csv')

## Summary
- All three baseline models trained and evaluated successfully.
- Metrics calculated: **MAE**, **RMSE**, **WAPE**, **sMAPE**.
- Results stored in `results/metrics/` and `results/predictions/`.
- Visualization plots saved in `results/plots/`.
- These baselines establish the performance floor for future advanced models.